# Practice Session 03: Management of networks data

<font size="+2" color="blue">Additional results: recipes</font>

Author: <font color="blue">Luca Franceschi</font>

E-mail: <font color="blue">luca.franceschi01@estudiant.upf.edu</font>

Date: <font color="blue">Due Oct. 12th, 18:30</font>

# 1. The flavors bi-partite graph

## 1.1. Read the bipartite graph in a dataframe


In [ ]:
# Feel free to add imports if you need them

import io
import csv
import pandas as pd
import networkx as nx

from networkx.algorithms import bipartite

import numpy as np
import matplotlib
import scipy

import itertools

from IPython.display import Image

In [ ]:
# Leave this code as-is

INPUT_INGR_FILENAME = "ingredients.tsv"
INPUT_COMP_FILENAME = "compounds.tsv"
INPUT_INGR_COMP_FILENAME = "ingredient-compound.tsv"

In [ ]:
# Leave this code as-is

ingredients = pd.read_csv(INPUT_INGR_FILENAME, sep="\t")
display(ingredients.head(3))

compounds = pd.read_csv(INPUT_COMP_FILENAME, sep="\t")
display(compounds.head(3))

ingr_comp = pd.read_csv(INPUT_INGR_COMP_FILENAME, sep="\t")
display(ingr_comp.head(3))


## 1.2. Create the flavors bipartite network

In [ ]:
# Create the flavors dataframe and show its first 20 rows

flavors = ingredients.set_index('ingredient_id').join(ingr_comp.set_index('ingredient_id'), how='inner')
flavors = flavors.set_index('compound_id').join(compounds.set_index('compound_id'), how='inner')
flavors = flavors.reset_index() # headers did not seem right
display(flavors.head(20))

In [ ]:
# Modify the flavors dataframe as explained above, and show its first 20 rows

flavors = flavors.drop(columns=['compound_code'])
flavors = flavors.sort_values(['ingredient_name', 'compound_name'])
flavors = flavors.reset_index(drop=True)
display(flavors.head(20))

In [ ]:
# Save flavors into a tab-separated file

flavors.to_csv('flavors.tsv', '\t', columns=['ingredient_name', 'ingredient_category', 'compound_name'], index=False)

## 1.3. Open this bi-partite network in Cytoscape


In [ ]:
# KEEP THIS CELL AS-IS

# Just adjust width/height if necessary

Image(url="flavors.png", width=1200)

In [ ]:
# KEEP THIS CELL AS-IS

# Just adjust width/height if necessary

Image(url="compounds-in-common.png", width=1200)

We can see that Garlic and Onion have in common 16 compounds, of which 10 seem to contain sulfur. A couple of those compounds are: propyl-disulfide, methyl_propyl_disulfide or allyl_methyl_trisulfide.

# 2. The ingredient-ingredient graph

## 2.1. Create an ingredient-ingredient.csv file


In [ ]:
# Create ingredients_array with the list of ingredients and to print the number of ingredients

ingredients_array = np.asarray(ingredients['ingredient_name'])
print("There are %d ingredients" % (len(ingredients_array)))

In [ ]:
# Create dictionary ingredient_to_compounds with a set of compounds for each ingredient. Print the number of 
# keys of this dictionary. It should be less than or equal to the number of ingredients

ingredient_to_compounds = {}

for index, row in flavors.iterrows():
	if row['ingredient_name'] not in ingredient_to_compounds:
		ingredient_to_compounds[row['ingredient_name']] = set()
	ingredient_to_compounds[row['ingredient_name']].add(row['compound_name'])

print('There are %d items in the dictionary' % len(ingredient_to_compounds))

In [ ]:
# Create the ingredient_ingredient graph

MIN_COMMON_COMPOUNDS = 70

ingredient_ingredient = nx.Graph()

for u, v in itertools.combinations(ingredients_array, 2):
	if u in ingredient_to_compounds and v in ingredient_to_compounds:
		weight = len(ingredient_to_compounds[u].intersection(ingredient_to_compounds[v]))
		if weight >= MIN_COMMON_COMPOUNDS:
			ingredient_ingredient.add_node(u)
			ingredient_ingredient.add_node(v)
			ingredient_ingredient.add_edge(u, v, weight=weight)

In [ ]:
# Leave as-is
print("The ingredient-ingredient graph has %d nodes and %d edges" %
      (ingredient_ingredient.number_of_nodes(), ingredient_ingredient.number_of_edges()))

In [ ]:
OUTPUT_INGR_INGR_FILENAME = 'ingredient-ingredient.gml'

In [ ]:
# Save graph G to file OUTPUT_INGR_INGR_FILENAME

nx.write_gml(ingredient_ingredient, OUTPUT_INGR_INGR_FILENAME)

## 2.2. Work with this file in Cytoscape

In [ ]:
# Change width if necessary

display(Image(url="ingr-ingr.png", width=1200))

display(Image(url="ingr-ingr-legend.gif", width=400))

Pairing 1: Coffee and peanut butter. We could see this pairing very often in a breakfast. We can see that these two ingredients have in common 83 compounds. Both are plant derivatives.

Pairing 2: Grilled beef and french fried potato. It is a very well-known pairing (steak fries) used in many cuisines with its origin in Belgium. We can see that these two ingredients have in common 86 compounds. In this case the french fries are a vegetable and grilled beef is meat.

It seems feasible that if two ingredients share a reasonable amount of compounds, they will have similar tastes or will be able to 'match' very well. Especially if the two ingredients are in a different category, such as pairing number two.

# Extra section

In [ ]:
import random

# Input: the whole string of the row
# Output: the list of ingredients
def string_formatting(row):
	raw_string = row.split(',') # split the string into a list
	raw_string.pop(0) # remove the first element of the list which is the region it is from
	return raw_string

# Input: the ingredients in a list and the filename of the output
# Output: saves the graph into gml file
def save_graph(row, filename, ingr_to_comp):
	graph = nx.Graph()
	graph.add_nodes_from(row)
	for ingredient in row: # iterate over all ingredients
		compounds_list = list(ingr_to_comp[ingredient])
		graph.add_nodes_from(compounds_list) # create all nodes regarding components of the ingredient
		for compound in ingr_to_comp[ingredient]:
			graph.add_edge(ingredient, compound)
	nx.write_gml(graph, filename)

# Input: the raw csv file, the amount of recipes that we want to extract and the random seed
def random_recipes_into_graphs(file_csv, amount=3, seed=0):
	random.seed(seed) # to always get the same results
	for i in range(amount):
		recipe_ingredients_str = file_csv.loc[random.randint(4, file_csv.index[-1])].tolist()[0] # extract a random row and converting it into a string
		row = string_formatting(recipe_ingredients_str)
		filename = 'recipe' + str(i) + '.gml'
		save_graph(row, filename, ingredient_to_compounds)

In [ ]:
recipes_csv = pd.read_csv('recipes.csv', sep='\t') # Import the recipes.csv as plain text, since we cannot format it directly

random_recipes_into_graphs(recipes_csv)

In [ ]:
display(Image(url="recipe0.png", width=1200))

In [ ]:
display(Image(url="recipe1.png", width=1200))

In [ ]:
display(Image(url="recipe2.png", width=1200))

We can see that the second and third recipes share many compounds, probably because the flavors they give match better. However in the first recipe we can see that there are two ingredients that do not share any compounds with any other ingredient (i.e.: olive oil and savory). In addition there are many compounds that do not share a big amount of compounds between them (e.g.: lavender, asparagus or basil). This probably means that the recipe may not taste very good...

# DELIVER (individually)

Read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/networks-science-course/blob/master/upf/upf-evaluation.md).

Deliver a zip file containing:

* This notebook
* The ``flavors.tsv``, ``flavors.png``, and ``flavors-legend.gif`` files
* The ``ingredient-ingredient.tsv``, ``ingr-ingr.png``, and ``ingr-ingr-legend.gif`` files

## Extra points available

For more learning and extra points, get the `recipes.csv` file. It contains one recipe per line, in this format:

```
EastAsian,roasted_sesame_seed,garlic,cayenne,seaweed,sesame_oil
```

This means there is one East Asian dish whose recipe requires the ingredients "roasted_sesame_seed", "garlic", "cayenne", "seaweed", and "sesame_oil".

Select 3 recipes and draw using Cytoscape a graph with their ingredients and the compounds in those ingredients. Include those subgraphs here, plus a brief commentary about whether the ingredients used share many compounds, few compounds, or not at all, and any other observations you want to make about the selected recipes.

**Note:** if you go for the extra points, add ``<font size="+2" color="blue">Additional results: recipes</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>
